<a href="https://colab.research.google.com/github/sneyx123-github/CopilotStudioSamples/blob/master/DrStop_DnsMapping_v1.0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The provided Python notebook efficiently manages Cloud Run deployments by handling region compatibility, existing state errors, and the 24-hour lock restriction for SSL challenges. A pre-flight DNS check using Python's socket library is recommended to enhance the tool's plug-and-play capability by verifying IP matches before gcloud execution. You can find the sample notebook at GitHub.

In [1]:
from google.colab import auth
PROJECT_ID = "project-46810a95-b6e5-47e4-adb"

# Melde dich mit deinem Google-Konto an
auth.authenticate_user()

# Setze das Projekt für die gcloud CLI
!gcloud config set project {PROJECT_ID}


[environment: untagged] Read more to tag: g.co/cloud/project-env-tag.
Updated property [core/project].


In [4]:
import time
from datetime import datetime

# WICHTIG: Nur die Subdomain verwenden, kein http:// oder ://
SERVICE = "gradio-test"
DOMAIN = "gradio-test.us.kommunikator-gmbh.com"
REGION = "europe-west1"

if 1:
  # 1. Altes Mapping löschen
  print(f"🗑️ Lösche altes Mapping für {DOMAIN}...")
  !gcloud beta run domain-mappings delete --domain {DOMAIN} --region {REGION} --quiet

  time.sleep(10)

  # 2. Mapping neu erstellen
  print(f"🚀 Erstelle Mapping neu für {DOMAIN}...")
  !gcloud beta run domain-mappings create --service {SERVICE} --domain {DOMAIN} --region {REGION}

# 3. Überwachungsschleife
print(f"⏳ Überwachung gestartet für {DOMAIN}. Prüfe alle 5 Minuten...")

while True:
    status_output = !gcloud beta run domain-mappings describe --domain {DOMAIN} --region {REGION} --format="yaml(status.conditions)"
    status_str = "\n".join(status_output)
    now = datetime.now().strftime('%H:%M:%S')

    if "status: 'True'\n    type: Ready" in status_str:
        print(f"✅ [{now}] ERFOLG! Deine App ist LIVE unter https://{DOMAIN}")
        break
    elif "message: Certificate issuance pending" in status_str:
        print(f"🔄 [{now}] Google prüft DNS... (Certificate issuance pending)")
    elif "reason: CertificatePending" in status_str:
        print(f"⏳ [{now}] Warte auf Zertifikat-Erstellung...")
    else:
        print(f"📡 [{now}] Status: {status_str[:100]}...") # Zeigt den Anfang des Status an

    time.sleep(300)

🗑️ Lösche altes Mapping für gradio-test.us.kommunikator-gmbh.com...
Mappings to [gradio-test.us.kommunikator-gmbh.com] now have been deleted.
🚀 Erstelle Mapping neu für gradio-test.us.kommunikator-gmbh.com...
Waiting for certificate provisioning. You must configure your DNS records for certificate issuance to begin.
NAME         RECORD TYPE  CONTENTS
gradio-test  CNAME        ghs.googlehosted.com.
⏳ Überwachung gestartet für gradio-test.us.kommunikator-gmbh.com. Prüfe alle 5 Minuten...
⏳ [19:25:21] Warte auf Zertifikat-Erstellung...
🔄 [19:30:24] Google prüft DNS... (Certificate issuance pending)
🔄 [19:35:27] Google prüft DNS... (Certificate issuance pending)
✅ [19:40:29] ERFOLG! Deine App ist LIVE unter https://gradio-test.us.kommunikator-gmbh.com


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
PROJECT_ID = "project-46810a95-b6e5-47e4-adb"
DOMAIN_SERVICE = "gradio-service.us.kommunikator-gmbh.com"
REGION = "europe-west1"

print(f"Describing domain mapping for {DOMAIN_SERVICE}:")
!gcloud beta run domain-mappings describe --domain {DOMAIN_SERVICE} --region {REGION} --project {PROJECT_ID} --format="yaml"

In [ ]:
PROJECT_ID = "project-46810a95-b6e5-47e4-adb"
DOMAIN_TEST = "gradio-test.us.kommunikator-gmbh.com"
REGION = "europe-west1"

print(f"Describing domain mapping for {DOMAIN_TEST}:")
!gcloud beta run domain-mappings describe --domain {DOMAIN_TEST} --region {REGION} --project {PROJECT_ID} --format="yaml"